# Lucas Kanade and KLT Tracking

In [1]:
import cv2
import numpy as np
import gdown

In [2]:
def read_video(url):
    file_id = url.split("/d/")[1].split("/")[0]
    download_url = f'https://drive.google.com/uc?id={file_id}'

    gdown.download(download_url,"video.mp4", quiet=False)

    return cv2.VideoCapture("video.mp4")

In [3]:
def play_video(cap):
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        cv2.imshow("Video Playback", frame)

        # Press 'q' to quit early
        if cv2.waitKey(25) & 0xFF == ord('q'):
            break
            
    cv2.destroyAllWindows()

In [4]:
cap = read_video("https://drive.google.com/file/d/16czHsa364vmD5lnAHgJRXSTAUFDWLrWH/view?usp=sharing")

Downloading...
From: https://drive.google.com/uc?id=16czHsa364vmD5lnAHgJRXSTAUFDWLrWH
To: C:\Users\sanka\PYTHON\COMPUTER VISION\video.mp4
100%|█████████████████████████████████████████████████████████████████████████████| 8.13M/8.13M [00:07<00:00, 1.13MB/s]


In [5]:
play_video(cap)

# Lucas Kanade for Two Frames

In [ ]:
def lucas_kanade_two_frames(cap):
    cap.set(cv2.CAP_PROP_POS_FRAMES,0)
    ret, old_frame = cap.read()
    if not ret:
        print("Failed to read the first frame")
        return

    old_gray = cv2.cvtColor(old_frame, cv2.COLOR_BGR2GRAY)
    features = cv2.goodFeaturesToTrack(old_gray, maxCorners=100, qualityLevel=0.3, minDistance=7, blockSize=7)

    lk_params = dict(winSize=(15, 15), maxLevel=2,
                     criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 10, 0.03))

    mask = np.zeros_like(old_frame)

    ret, new_frame = cap.read()
    if not ret:
        print("Failed to read the second frame")
        return

    new_gray = cv2.cvtColor(new_frame, cv2.COLOR_BGR2GRAY)
    next_points, status, _ = cv2.calcOpticalFlowPyrLK(old_gray, new_gray, features, None, **lk_params)

    good_new = next_points[status == 1]
    good_old = features[status == 1]

    for i, (new, old) in enumerate(zip(good_new, good_old)):
        x_new, y_new = new.ravel()
        x_old, y_old = old.ravel()
        mask = cv2.line(mask, (int(x_new), int(y_new)), (int(x_old), int(y_old)), (0, 255, 0), 2)
        new_frame = cv2.circle(new_frame, (int(x_new), int(y_new)), 5, (0, 0, 255), -1)

    output = cv2.add(new_frame, mask)
    cv2.imshow('Lucas-Kanade Optical Flow (First Two Frames)', output)

    cv2.waitKey(0)
    cv2.destroyAllWindows()

lucas_kanade_two_frames(cap)

# Lucas Kanade and KLT for Video

In [7]:
ret, old_frame = cap.read()
old_gray = cv2.cvtColor(old_frame, cv2.COLOR_BGR2GRAY)

features = cv2.goodFeaturesToTrack(old_gray, maxCorners=100, qualityLevel=0.3, minDistance=7, blockSize=7)

lk_params = dict(winSize=(15, 15), maxLevel=2,
                 criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 10, 0.03))

mask = np.zeros_like(old_frame)

In [8]:
cap.set(cv2.CAP_PROP_POS_FRAMES,0)
while True:
    ret, frame = cap.read()
    if not ret:
        print('Video Finishes')
        break

    frame_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    next_points, status, _ = cv2.calcOpticalFlowPyrLK(old_gray, frame_gray, features, None, **lk_params)

    good_new = next_points[status == 1]
    good_old = features[status == 1]

    for i, (new, old) in enumerate(zip(good_new, good_old)):
        x_new, y_new = new.ravel()
        x_old, y_old = old.ravel()
        mask = cv2.line(mask, (int(x_new), int(y_new)), (int(x_old), int(y_old)), (0, 255, 0), 2)
        frame = cv2.circle(frame, (int(x_new), int(y_new)), 5, (0, 0, 255), -1)

    output = cv2.add(frame, mask)
    cv2.imshow('KLT Tracking', output)

    old_gray = frame_gray.copy()
    features = good_new.reshape(-1, 1, 2)

    if cv2.waitKey(30) & 0xFF == ord('q'):
        break

cv2.destroyAllWindows()